# 02 — Model Comparison & Visualization

Compare bias evaluation results across models with interactive visualizations.

**Prerequisites:** Run `make benchmarks` first to generate results.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

## 1. Load Results
Load benchmark results from JSON (generated by `make benchmarks`).

In [ ]:
results_path = Path('../reports/benchmark_results/all_results.json')

if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f'Loaded {len(results)} benchmark results')
    for r in results:
        print(f'  {r.get("model", "?")} / {r.get("benchmark", "?")}')
else:
    print('No results found. Run: make benchmarks')
    results = []

## 2. Comparison Matrix

In [ ]:
matrix_path = Path('../reports/model_comparison_matrix.csv')

if matrix_path.exists():
    df = pd.read_csv(matrix_path)
    display(df)
else:
    print('No comparison matrix found. Run benchmarks first.')

## 3. Heatmap Visualization

In [ ]:
if results:
    from src.utils.visualization import create_model_comparison_heatmap
    fig = create_model_comparison_heatmap()
    fig.show()
else:
    print('Run benchmarks to generate heatmap data.')

## 4. Stereotype Radar Chart

In [ ]:
if results:
    from src.utils.visualization import create_stereotype_radar
    fig = create_stereotype_radar(results)
    if fig:
        fig.show()
    else:
        print('No StereoSet results available.')

## 5. Sentiment Disparity Chart

In [ ]:
if results:
    from src.utils.visualization import create_sentiment_disparity_chart
    fig = create_sentiment_disparity_chart(results)
    if fig:
        fig.show()
    else:
        print('No sentiment disparity results available.')

## 6. Counterfactual Fairness Demo
Test how prompts change when demographics are swapped.

In [ ]:
from src.guardrails_app.mitigation import CounterfactualAugmenter

cf = CounterfactualAugmenter()

test_prompts = [
    'The man was promoted to CEO.',
    'The White student excelled in academics.',
    'The Christian community organized a charity event.',
]

for prompt in test_prompts:
    print(f'\nOriginal: "{prompt}"')
    for category in ['gender', 'race', 'religion']:
        cfs = cf.generate_counterfactuals(prompt, category)
        for c in cfs[:3]:
            print(f'  [{category}] {c["swapped"]}: "{c["counterfactual"]}"')

## 7. Fairness Card Generation

In [ ]:
from compliance.fairness_card import FairnessCard

card_gen = FairnessCard('llama3-8b')
card = card_gen.generate(
    benchmark_results=results if results else None,
)

print(json.dumps(card['compliance'], indent=2))
print()
print('Recommendations:')
for rec in card['recommendations']:
    print(f'  - {rec}')